# Загрузка библиотек

In [33]:
import pandas as pd
import sys
from sklearn.model_selection import train_test_split
from sklearn.linear_model  import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [2]:
sys.path.append('../utils_functionality/')
from data_analysis.split_analisys import *
from data_analysis.reports import *
from models.class_models import *
from models.class_models import *
from models.hyperparams_tuning import *
from models.blending import *
from models.alternative_models import *

Imported 0.2.6 version. Select nrows to a small number when running on huge datasets.
output = featurewiz(dataname, target, corr_limit=0.90, verbose=2, sep=',', 
		header=0, test_data='',feature_engg='', category_encoders='',
		dask_xgboost_flag=False, nrows=None, skip_sulov=False)
Create new features via 'feature_engg' flag : ['interactions','groupby','target']



In [3]:
import wandb
wandb.login(key='8b1f2ae49570ea59dc8612ff434ca497d463d221')

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: dimazharikov10 (dstech). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\dimaz/.netrc


True

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
RANDOM_STATE = 42

# Загрузка данных и разделение на train, test

In [6]:
df = pd.read_excel('../data/df_merged_edited.xlsx', index_col=[0])
stranger_things = ['voltage', 'long_impulse_duration', 'long_impulse_dur_binary']
train, test = train_test_split(df, test_size=0.33, random_state=RANDOM_STATE)

# Logistic Regression

## splashing

In [7]:
model = LogisticRegression()

In [8]:
cs = CreateSamples(df, train, test, target='splashing', use_featurewiz=True, strange_columns=stranger_things)
X_train, X_test, y_train, y_test = cs.get_samples()

############################################################################################
############       F A S T   F E A T U R E  E N G G    A N D    S E L E C T I O N ! ########
# Be judicious with featurewiz. Don't use it to create too many un-interpretable features! #
############################################################################################
featurewiz has selected 0.7 as the correlation limit. Change this limit to fit your needs...
Skipping feature engineering since no feature_engg input...
Skipping category encoding since no category encoders specified in input...
#### Single_Label Binary_Classification problem ####
    Loaded train data. Shape = (372, 20)
#### Single_Label Binary_Classification problem ####
No test data filename given...
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#############################################

In [9]:
X_train.head()

,surface_tension,particle_liquid_density_ratio,We,roughness,Re,liquid_density,wettability
39,0.0691,0.877193,833.534028,0.04,2110.385675,1140,2
44,0.0691,0.877193,798.846587,10.89,2022.562172,1140,0
176,0.0679,1.864407,891.662791,2.49,661.664919,1180,1
201,0.0679,0.847458,875.302006,0.10,649.524278,1180,0
186,0.0269,1.219512,1465.995744,0.04,1466.205641,820,2


In [10]:
numeric_features = ['surface_tension', 'roughness', 'particle_mean_diameter', 'viscosity']
categorical_features = ['inclination', 'wettability']

In [ ]:
wandb.init(
    project='drop-wall-impact-clfr', 
    name='featurewiz + LogReg, splashing, no strange columns'
)

In [12]:
clf = create_pipeline(numeric_features, categorical_features, model)
model.fit(X_train, y_train)
y_preds = model.predict(X_test)
y_preds_proba = model.predict_proba(X_test)[:, 1]
roc, acc, f1 = get_classification_report(y_train, y_test, y_preds, y_preds_proba, return_metrics=True)
wandb.log({
    'accuracy': acc,
    'ROC-AUC': roc,
    'F1-score': f1
    })

,precision,recall,f1-score,support
0,0.632911,0.769231,0.694444,65.000000
1,0.659091,0.500000,0.568627,58.000000
accuracy,0.642276,0.642276,0.642276,0.642276
macro avg,0.646001,0.634615,0.631536,123.000000
weighted avg,0.645256,0.642276,0.635116,123.000000


## net_impact

In [13]:
model = LogisticRegression()

In [14]:
cs = CreateSamples(df, train, test, target='net_impact', use_featurewiz=True, strange_columns=stranger_things)
X_train, X_test, y_train, y_test = cs.get_samples()

############################################################################################
############       F A S T   F E A T U R E  E N G G    A N D    S E L E C T I O N ! ########
# Be judicious with featurewiz. Don't use it to create too many un-interpretable features! #
############################################################################################
featurewiz has selected 0.7 as the correlation limit. Change this limit to fit your needs...
Skipping feature engineering since no feature_engg input...
Skipping category encoding since no category encoders specified in input...
#### Single_Label Binary_Classification problem ####
    Loaded train data. Shape = (372, 20)
#### Single_Label Binary_Classification problem ####
No test data filename given...
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#############################################

In [15]:
X_train.head()

,surface_tension,roughness,particle_mean_diameter,inclination,viscosity,wettability
39,0.0691,0.04,0.000041,0,0.00689,2
44,0.0691,10.89,0.000041,0,0.00689,0
176,0.0679,2.49,0.000041,0,0.02310,1
201,0.0679,0.10,0.000041,0,0.02310,0
186,0.0269,0.04,0.000041,0,0.00679,2


In [16]:
numeric_features = ['surface_tension', 'roughness', 'particle_mean_diameter', 'viscosity']
categorical_features = ['inclination', 'wettability']

In [ ]:
wandb.init(
    project='drop-wall-impact-clfr', 
    name='featurewiz + LogReg, net_impact, no strange columns'
)

In [18]:
clf = create_pipeline(numeric_features, categorical_features, model)
model.fit(X_train, y_train)
y_preds = model.predict(X_test)
y_preds_proba = model.predict_proba(X_test)[:, 1]
roc, acc, f1 = get_classification_report(y_train, y_test, y_preds, y_preds_proba, return_metrics=True)
wandb.log({
    'accuracy': acc,
    'ROC-AUC': roc,
    'F1-score': f1
    })

,precision,recall,f1-score,support
0,0.666667,1.000000,0.800000,82.000000
1,0.000000,0.000000,0.000000,41.000000
accuracy,0.666667,0.666667,0.666667,0.666667
macro avg,0.333333,0.500000,0.400000,123.000000
weighted avg,0.444444,0.666667,0.533333,123.000000


# SVM

## splashing

In [25]:
model = SVC(probability=True)

In [20]:
cs = CreateSamples(df, train, test, target='splashing', use_featurewiz=True, strange_columns=stranger_things)
X_train, X_test, y_train, y_test = cs.get_samples()

############################################################################################
############       F A S T   F E A T U R E  E N G G    A N D    S E L E C T I O N ! ########
# Be judicious with featurewiz. Don't use it to create too many un-interpretable features! #
############################################################################################
featurewiz has selected 0.7 as the correlation limit. Change this limit to fit your needs...
Skipping feature engineering since no feature_engg input...
Skipping category encoding since no category encoders specified in input...
#### Single_Label Binary_Classification problem ####
    Loaded train data. Shape = (372, 20)
#### Single_Label Binary_Classification problem ####
No test data filename given...
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#############################################

In [21]:
X_train.head()

,surface_tension,particle_liquid_density_ratio,We,roughness,Re,liquid_density,wettability
39,0.0691,0.877193,833.534028,0.04,2110.385675,1140,2
44,0.0691,0.877193,798.846587,10.89,2022.562172,1140,0
176,0.0679,1.864407,891.662791,2.49,661.664919,1180,1
201,0.0679,0.847458,875.302006,0.10,649.524278,1180,0
186,0.0269,1.219512,1465.995744,0.04,1466.205641,820,2


In [22]:
numeric_features = ['surface_tension', 'roughness', 'particle_mean_diameter', 'viscosity']
categorical_features = ['inclination', 'wettability']

In [ ]:
wandb.init(
    project='drop-wall-impact-clfr', 
    name='featurewiz + SVC, splashing, no strange columns'
)

In [26]:
clf = create_pipeline(numeric_features, categorical_features, model)
model.fit(X_train, y_train)
y_preds = model.predict(X_test)
y_preds_proba = model.predict_proba(X_test)[:, 1]
roc, acc, f1 = get_classification_report(y_train, y_test, y_preds, y_preds_proba, return_metrics=True)
wandb.log({
    'accuracy': acc,
    'ROC-AUC': roc,
    'F1-score': f1
    })

,precision,recall,f1-score,support
0,0.616279,0.815385,0.701987,65.000000
1,0.675676,0.431034,0.526316,58.000000
accuracy,0.634146,0.634146,0.634146,0.634146
macro avg,0.645977,0.623210,0.614151,123.000000
weighted avg,0.644287,0.634146,0.619150,123.000000


## net_impact

In [27]:
model = SVC(probability=True)

In [28]:
cs = CreateSamples(df, train, test, target='net_impact', use_featurewiz=True, strange_columns=stranger_things)
X_train, X_test, y_train, y_test = cs.get_samples()

############################################################################################
############       F A S T   F E A T U R E  E N G G    A N D    S E L E C T I O N ! ########
# Be judicious with featurewiz. Don't use it to create too many un-interpretable features! #
############################################################################################
featurewiz has selected 0.7 as the correlation limit. Change this limit to fit your needs...
Skipping feature engineering since no feature_engg input...
Skipping category encoding since no category encoders specified in input...
#### Single_Label Binary_Classification problem ####
    Loaded train data. Shape = (372, 20)
#### Single_Label Binary_Classification problem ####
No test data filename given...
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#############################################

In [29]:
X_train.head()

,surface_tension,roughness,particle_mean_diameter,inclination,viscosity,wettability
39,0.0691,0.04,0.000041,0,0.00689,2
44,0.0691,10.89,0.000041,0,0.00689,0
176,0.0679,2.49,0.000041,0,0.02310,1
201,0.0679,0.10,0.000041,0,0.02310,0
186,0.0269,0.04,0.000041,0,0.00679,2


In [30]:
numeric_features = ['surface_tension', 'roughness', 'particle_mean_diameter', 'viscosity']
categorical_features = ['inclination', 'wettability']

In [ ]:
wandb.init(
    project='drop-wall-impact-clfr', 
    name='featurewiz + SVC, net_impact, no strange columns'
)

In [32]:
clf = create_pipeline(numeric_features, categorical_features, model)
model.fit(X_train, y_train)
y_preds = model.predict(X_test)
y_preds_proba = model.predict_proba(X_test)[:, 1]
roc, acc, f1 = get_classification_report(y_train, y_test, y_preds, y_preds_proba, return_metrics=True)
wandb.log({
    'accuracy': acc,
    'ROC-AUC': roc,
    'F1-score': f1
    })

,precision,recall,f1-score,support
0,0.666667,1.000000,0.800000,82.000000
1,0.000000,0.000000,0.000000,41.000000
accuracy,0.666667,0.666667,0.666667,0.666667
macro avg,0.333333,0.500000,0.400000,123.000000
weighted avg,0.444444,0.666667,0.533333,123.000000


# KNeighborsClassifier

## splashing

In [34]:
model = KNeighborsClassifier()

In [35]:
cs = CreateSamples(df, train, test, target='splashing', use_featurewiz=True, strange_columns=stranger_things)
X_train, X_test, y_train, y_test = cs.get_samples()

############################################################################################
############       F A S T   F E A T U R E  E N G G    A N D    S E L E C T I O N ! ########
# Be judicious with featurewiz. Don't use it to create too many un-interpretable features! #
############################################################################################
featurewiz has selected 0.7 as the correlation limit. Change this limit to fit your needs...
Skipping feature engineering since no feature_engg input...
Skipping category encoding since no category encoders specified in input...
#### Single_Label Binary_Classification problem ####
    Loaded train data. Shape = (372, 20)
#### Single_Label Binary_Classification problem ####
No test data filename given...
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#############################################

In [38]:
numeric_features = ['surface_tension', 'particle_liquid_density_ratio', 'We', 'roughness',
       'Re', 'liquid_density']
categorical_features = ['wettability']

In [ ]:
wandb.init(
    project='drop-wall-impact-clfr', 
    name='featurewiz + KNN, splashing, no strange columns'
)

In [40]:
clf = create_pipeline(numeric_features, categorical_features, model)
model.fit(X_train, y_train)
y_preds = model.predict(X_test)
y_preds_proba = model.predict_proba(X_test)[:, 1]
roc, acc, f1 = get_classification_report(y_train, y_test, y_preds, y_preds_proba, return_metrics=True)
wandb.log({
    'accuracy': acc,
    'ROC-AUC': roc,
    'F1-score': f1
    })

,precision,recall,f1-score,support
0,0.712329,0.800000,0.753623,65.000000
1,0.740000,0.637931,0.685185,58.000000
accuracy,0.723577,0.723577,0.723577,0.723577
macro avg,0.726164,0.718966,0.719404,123.000000
weighted avg,0.725377,0.723577,0.721352,123.000000


## net_impact

In [41]:
model = KNeighborsClassifier()

In [42]:
cs = CreateSamples(df, train, test, target='net_impact', use_featurewiz=True, strange_columns=stranger_things)
X_train, X_test, y_train, y_test = cs.get_samples()

############################################################################################
############       F A S T   F E A T U R E  E N G G    A N D    S E L E C T I O N ! ########
# Be judicious with featurewiz. Don't use it to create too many un-interpretable features! #
############################################################################################
featurewiz has selected 0.7 as the correlation limit. Change this limit to fit your needs...
Skipping feature engineering since no feature_engg input...
Skipping category encoding since no category encoders specified in input...
#### Single_Label Binary_Classification problem ####
    Loaded train data. Shape = (372, 20)
#### Single_Label Binary_Classification problem ####
No test data filename given...
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#############################################

In [43]:
X_train.head()

,surface_tension,roughness,particle_mean_diameter,inclination,viscosity,wettability
39,0.0691,0.04,0.000041,0,0.00689,2
44,0.0691,10.89,0.000041,0,0.00689,0
176,0.0679,2.49,0.000041,0,0.02310,1
201,0.0679,0.10,0.000041,0,0.02310,0
186,0.0269,0.04,0.000041,0,0.00679,2


In [45]:
numeric_features = ['surface_tension', 'roughness', 'particle_mean_diameter','viscosity']
categorical_features = ['inclination', 'wettability']

In [ ]:
wandb.init(
    project='drop-wall-impact-clfr', 
    name='featurewiz + KNN, net_impact, no strange columns'
)

In [47]:
clf = create_pipeline(numeric_features, categorical_features, model)
model.fit(X_train, y_train)
y_preds = model.predict(X_test)
y_preds_proba = model.predict_proba(X_test)[:, 1]
roc, acc, f1 = get_classification_report(y_train, y_test, y_preds, y_preds_proba, return_metrics=True)
wandb.log({
    'accuracy': acc,
    'ROC-AUC': roc,
    'F1-score': f1
    })

,precision,recall,f1-score,support
0,0.855422,0.865854,0.860606,82.000000
1,0.725000,0.707317,0.716049,41.000000
accuracy,0.813008,0.813008,0.813008,0.813008
macro avg,0.790211,0.786585,0.788328,123.000000
weighted avg,0.811948,0.813008,0.812421,123.000000
